In [1]:
# Parameters
ra = 0
sign = 0


In [2]:
ra_parameter = 0
sign_parameter = 0

In [3]:
# Data manipulation and analysis
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

# Astropy modules
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.visualization import simple_norm
from astropy.wcs import WCS
from astropy.convolution import Gaussian1DKernel, convolve_fft

from astropy.coordinates import SkyCoord

# Matplotlib for plotting
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Scipy and related modules
import scipy.sparse as sp

# Spectral cube handling
from spectral_cube import SpectralCube as sc

# Progress bar
from tqdm import trange

# Additional libraries
from kneed import KneeLocator
from photutils.aperture import SkyEllipticalAperture
from regions import EllipseSkyRegion

# Multithreading
from joblib import Parallel, delayed

# Suppress warnings from spectral_cube
import warnings
from spectral_cube.utils import PossiblySlowWarning
warnings.filterwarnings("ignore", category=PossiblySlowWarning)

# Set global options for numpy and matplotlib
np.seterr(divide="ignore", invalid="ignore")
np.set_printoptions(precision=10, suppress=True)
plt.rcParams.update({'figure.max_open_warning': 0, 'font.size': 12})
delta_min = 1e-12

%matplotlib widget

In [4]:
# Define the source and target cube patterns

crafts_pattern = "./CRAFTS/South/Baseline Corrected/CRAFTS_RA{ra}_DEC-13_2_cor_{sign}.fits"
hi4pi_pattern = "./HI4PI/HI4PI_RA{ra}_DEC-13_2.fits"
output_pattern = "./Figures/CRAFTS_RA{ra}_DEC-13_2_{sign}_moment{m}.fits"
catalog_pattern = "./Catalogs/CRAFTS_RA{ra}_DEC-13_2_cor_{sign}.csv"

# Define RA ranges and signs

ra_ranges = ["60_80", "80_100", "100_120", "120_140"]
signs = ["+", "-"]
moments = [0, 1, 2]

ra = ra_ranges[ra_parameter]
sign = signs[sign_parameter]

crafts_file = crafts_pattern.format(ra=ra, sign=sign)
hi4pi_file = hi4pi_pattern.format(ra=ra, sign=sign)
catalog_file = catalog_pattern.format(ra=ra, sign=sign)
output_file = []
for m in moments:
    output_file.append(output_pattern.format(ra=ra, sign=sign, m=m))

In [5]:
# CRAFTS cube

cube = sc.read(crafts_file).with_spectral_unit(u.km / u.s)
# cube = cube.spectral_slab(-200 * u.km / u.s, -55 * u.km / u.s)
data = cube.unmasked_data[:, :, :].value
hdr = cube.header
wcs = WCS(hdr)
pixel_to_deg = hdr["CDELT2"]

# data mask
# data[:,:,0:201] = np.nan

MEDIAN = np.nanmedian(data)
STD = np.nanstd(data)
print("Mean:", MEDIAN)
print("Std:", STD)

data = np.nan_to_num(data)
velocities = cube.spectral_axis.value
delta_v = np.abs(velocities[1] - velocities[0])  # velocity resolution in km/s
print("CRAFTS cube\n", cube)
print("delta_v =", delta_v, "km/s\n")

# Uncomment the following lines if you want to average every two slices

# stack = data.shape[0]
# for i in range(0, stack, 2):
#     if i // 2 < stack // 2:
#         data[i // 2] = (data[i] + data[i + 1]) / 2
#
# data = data[: stack // 2]

# HI4PI cube

hi4pi_cube = sc.read(hi4pi_file).with_spectral_unit(u.km / u.s)
hi4pi_cube = hi4pi_cube.spectral_slab(np.min(velocities) * u.km / u.s, np.max(velocities) * u.km / u.s)
hidpi_data = hi4pi_cube.unmasked_data[:, :, :].value
hi4pi_velocities = hi4pi_cube.spectral_axis.value
hi4pi_delta_v = np.abs(hi4pi_velocities[1] - hi4pi_velocities[0])
print("HI4PI cube\n", hi4pi_cube)
print("hi4pi_delta_v =", hi4pi_delta_v, "km/s")

KeyboardInterrupt: 

In [ ]:
# sigma clipping data to select value v > sσ


def test_sigma_clip(test=True):
    if test == True:
        # use knee/elbow value to find the best s value
        s_ = np.arange(1, 5.2, 0.2)
        y_ = []

        for i in trange(len(s_)):
            upper_bound = MEDIAN + STD * s_[i]
            mask = data > upper_bound  # mask == True: v > sσ
            y_.append(
                np.count_nonzero(mask) / np.count_nonzero(data)
            )  # N(True) / N(total)

        kneedle = KneeLocator(
            s_, y_, S=1.0, curve="convex", direction="decreasing"
        )  # knee/elbow value
        print("Best value of x =", round(kneedle.elbow, 4))
        print("N(>xσ) / N(total) =", round(kneedle.elbow_y, 4))

        # plot the results as in kneedle.plot_knee()
        fig = plt.figure()
        plt.scatter(s_, y_, label="N(>xσ) / N(total)")
        plt.scatter(s_, kneedle.y_difference, label="Difference")
        plt.plot(s_, y_)
        plt.plot(s_, kneedle.y_difference)
        plt.ylim(-0.05, 0.65)
        plt.vlines(round(kneedle.elbow, 3), 0, 1, ls="--", colors="red", label="Elbow")
        plt.xlabel("x")
        plt.ylabel("N(>xσ) / N(total)")
        plt.grid(axis="y", ls="--")
        plt.legend()
        plt.show()
        return kneedle.elbow
    else:
        # choose s = 2
        return 2


s = test_sigma_clip(test=True)

In [ ]:
upper_bound = MEDIAN + STD * s
mask = data > upper_bound

# index is a 2D array, where each row is [v, y, x] coordinate for a point with value v > sσ
# this will be used for clustering
index = np.argwhere(mask)

# DBSCAN

In [ ]:
# Set the parameters for DBSCAN: eps, minPts

eps = np.sqrt(3)
minPts = 12

In [ ]:
# DBSCAN clustering

db = DBSCAN(eps=eps, min_samples=minPts, n_jobs=-1).fit(index)  # DBSCAN clustering
# db.labels_ is an array of the same length as index, where each element is the cluster label for that point.
labels = db.labels_

# Number of clusters in labels, ignoring noise if present.
unique_labels = set(labels)
n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print("Estimated number of clusters: %d" % n_clusters)
print("Estimated number of noise points: %d" % n_noise)

# 计算 Moment 0, 1, 2

In [ ]:
# Calculate moment 0 using pure Python (faster for smaller clusters)
def moment_0_py(data, vyx):
    # vyx is coordinates [v, y, x] of points in the cluster (labels == cluster_id)
    # Extract the velocity column and the coordinate column separately
    velo = vyx[:, 0]  # velocity [v]
    coords = vyx[:, 1:]  # coordinates [y, x]
    # Find unique rows in the x,y columns and their indices
    unique_coords, indices = np.unique(coords, axis=0, return_inverse=True)
    # unique_coords is a 2D array of unique [y, x] coordinates
    # indices is an array of the same length as coords, where each element is the index of the unique coordinate
    # This means that for each coordinate in coords, we can find several velocities in velo that correspond to it.

    # Calculate moment 0 of the velocity axis
    # for each unique pair of the coordinates
    def calc_moment_0(unique_coords, indices, velo, data):
        moment_0 = np.zeros(len(unique_coords))
        for i in range(len(unique_coords)):  # for each unique coordinate
            # Calculate the moment 0 for this coordinate
            # M0 = \int I_v dv = \sum I_v * delta_v
            m0 = 0
            coo = unique_coords[i]
            for v in velo[
                indices == i
            ]:  # sum over all velocities that correspond to this coordinate
                m0 += data[v, coo[0], coo[1]] * abs(delta_v)  # moment 0
            moment_0[i] = m0
        return moment_0

    moment_0 = calc_moment_0(unique_coords, indices, velo, data)
    # Combine the unique coordinates with their moment 0
    # wrap the result in a sparse matrix (COO format)
    # this is more memory efficient for large datasets
    moment_0_coo = sp.coo_array(
        (moment_0, (unique_coords[:, 0], unique_coords[:, 1])), shape=data[0].shape
    )
    return moment_0_coo


# Calculate moment 0 using pure Spectral-Cube (faster for larger clusters)
# https://github.com/radio-astro-tools/spectral-cube/blob/e98b6c3c05e3a21c6ca62524e1dea9582ad5cd38/spectral_cube/_moments.py#L170
def moment_0_sc(data, vyx):
    # create a cube which only contains the values of points in the cluster (labels == cluster_id)
    bool_array = np.zeros(data.shape, dtype=np.bool)
    bool_array[vyx[:, 0], vyx[:, 1], vyx[:, 2]] = (
        True  # mark all points in the cluster (labels == cluster_id) as True
    )
    cluster_data = (
        bool_array * data
    )  # multiply the data with the boolean array to get only the values of points in the cluster
    bool_cube = sc(cluster_data, wcs=wcs).with_spectral_unit(u.km / u.s)
    # Calculate moment 0 of the velocity axis
    moment_0 = bool_cube.moment(order=0).value
    # wrap the result in a sparse matrix (COO format)
    # this is more memory efficient for large datasets
    moment_0_coo = sp.coo_array(moment_0)
    return moment_0_coo


def moment_0_func(data, index, labels, n_clusters):
    # Optimized moment_0_func with parallel processing for faster computation.
    def compute_moment_0_for_cluster(cluster_id):
        vyx = index[
            labels == cluster_id
        ]  # Get all coordinates [v, y, x] of points in the cluster (labels == cluster_id)
        size = len(vyx)
        if size < 100000:
            return moment_0_py(
                data, vyx
            )  # Use Python implementation for smaller clusters
        else:
            return moment_0_sc(data, vyx)  # Use spectral-cube for larger clusters

    # Use parallel processing to compute moment 0 for each cluster
    moment_0_results = Parallel(n_jobs=-1, backend="threading")(
        delayed(compute_moment_0_for_cluster)(cluster_id)
        for cluster_id in trange(n_clusters)
    )

    # Combine results into a single array
    moment_0_cube = np.hstack(moment_0_results)
    return moment_0_cube

In [ ]:
moment_0_cube = moment_0_func(data, index, labels, n_clusters)

In [ ]:
# Calculate moment 1 using pure Python (faster for smaller clusters)
def moment_1_py(data, vyx):
    # vyx is coordinates [v, y, x] of points in the cluster (labels == cluster_id)
    # Extract the velocity column and the coordinate column separately
    velo = vyx[:, 0]  # velocity [v]
    coords = vyx[:, 1:]  # coordinates [y, x]
    # Find unique rows in the x,y columns and their indices
    unique_coords, indices = np.unique(coords, axis=0, return_inverse=True)
    # unique_coords is a 2D array of unique [y, x] coordinates
    # indices is an array of the same length as coords, where each element is the index of the unique coordinate
    # This means that for each coordinate in coords, we can find several velocities in velo that correspond to it.

    # Calculate moment 1 of the velocity axis
    # for each unique pair of the coordinates
    def calc_moment_1(unique_coords, indices, velo, data):
        moment_1 = np.zeros(len(unique_coords))
        for i in range(len(unique_coords)):  # for each unique coordinate
            # Calculate the moment 1 for this coordinate
            # M1 = \int v I_v dv / M0 = \sum I_v * v / \sum I_v
            m0 = 0
            m1 = 0
            coo = unique_coords[i]
            for v in velo[
                indices == i
            ]:  # sum over all velocities that correspond to this coordinate
                m0 += data[v, coo[0], coo[1]]
                vel = velocities[v]
                m1 += data[v, coo[0], coo[1]] * vel  # moment 1
            moment_1[i] = m1 / m0
        return moment_1

    moment_1 = calc_moment_1(unique_coords, indices, velo, data)
    # Combine the unique coordinates with their moment 0
    # wrap the result in a sparse matrix (COO format)
    # this is more memory efficient for large datasets
    moment_1_coo = sp.coo_array(
        (moment_1, (unique_coords[:, 0], unique_coords[:, 1])), shape=data[0].shape
    )
    return moment_1_coo


# Calculate moment 0 using pure Spectral-Cube (faster for larger clusters)
# https://github.com/radio-astro-tools/spectral-cube/blob/e98b6c3c05e3a21c6ca62524e1dea9582ad5cd38/spectral_cube/_moments.py#L170
def moment_1_sc(data, vyx):
    # create a cube which only contains the values of points in the cluster (labels == cluster_id)
    bool_array = np.zeros(data.shape, dtype=np.bool)
    bool_array[vyx[:, 0], vyx[:, 1], vyx[:, 2]] = (
        True  # mark all points in the cluster (labels == cluster_id) as True
    )
    cluster_data = (
        bool_array * data
    )  # multiply the data with the boolean array to get only the values of points in the cluster
    bool_cube = sc(cluster_data, wcs=wcs).with_spectral_unit(u.km / u.s)
    # Calculate moment 1 of the velocity axis
    moment_1 = bool_cube.moment(order=1).value
    # wrap the result in a sparse matrix (COO format)
    # this is more memory efficient for large datasets
    moment_1_coo = sp.coo_array(moment_1)
    return moment_1_coo


def moment_1_func(data, index, labels, n_clusters):
    # Optimized moment_1_func with parallel processing for faster computation.
    def compute_moment_1_for_cluster(cluster_id):
        vyx = index[
            labels == cluster_id
        ]  # Get all coordinates [v, y, x] of points in the cluster (labels == cluster_id)
        size = len(vyx)
        if size < 100000:
            return moment_0_py(
                data, vyx
            )  # Use Python implementation for smaller clusters
        else:
            return moment_0_sc(data, vyx)  # Use spectral-cube for larger clusters

    # Use parallel processing to compute moment 1 for each cluster
    moment_1_results = Parallel(n_jobs=-1, backend="threading")(
        delayed(compute_moment_1_for_cluster)(cluster_id)
        for cluster_id in trange(n_clusters)
    )

    # Combine results into a single array
    moment_1_cube = np.hstack(moment_1_results)
    return moment_1_cube

In [ ]:
moment_1_cube = moment_1_func(data, index, labels, n_clusters)

In [ ]:
# Calculate moment 2 (FWHM) using pure Python (faster for smaller clusters)
def moment_2_py(data, vyx):
    # vyx is coordinates [v, y, x] of points in the cluster (labels == cluster_id)
    # Extract the velocity column and the coordinate column separately
    velo = vyx[:, 0]  # velocity [v]
    coords = vyx[:, 1:]  # coordinates [y, x]
    # Find unique rows in the x,y columns and their indices
    unique_coords, indices = np.unique(coords, axis=0, return_inverse=True)
    # unique_coords is a 2D array of unique [y, x] coordinates
    # indices is an array of the same length as coords, where each element is the index of the unique coordinate
    # This means that for each coordinate in coords, we can find several velocities in velo that correspond to it.

    # Calculate moment 2 of the velocity axis
    # for each unique pair of the coordinates
    def calc_moment_2(unique_coords, indices, velo, data):
        moment_2 = np.zeros(len(unique_coords))
        for i in range(len(unique_coords)):  # for each unique coordinate
            # Calculate the moment 1 for this coordinate
            # M2 = \int I_v (v - M1)^2 dv / M0 = \sum I_v * (v - M1)^2 / \sum I_v
            m0 = 0
            m1 = 0
            m2 = 0
            coo = unique_coords[i]
            # Calculate moment 0 and moment 1 for this coordinate
            for v in velo[
                indices == i
            ]:  # sum over all velocities that correspond to this coordinate
                m0 += data[v, coo[0], coo[1]]
                vel = velocities[v]
                m1 += data[v, coo[0], coo[1]] * vel  # moment 1
            m1 = m1 / m0
            # Calculate moment 2 for this coordinate
            for v in velo[indices == i]:
                vel = velocities[v]
                m2 += data[v, coo[0], coo[1]] * ((vel - m1) ** 2)  # moment 2
            m2 = m2 / m0
            # Convert moment 2 to FWHM
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("error")
                    sigma = np.sqrt(m2)  # linewidth_sigma
            except RuntimeWarning as e:
                warnings.simplefilter("ignore", category=RuntimeWarning)
                sigma = np.sqrt(m2)  # linewidth_sigma, warning (m2 < 0)
            fwhm = sigma * np.sqrt(8 * np.log(2))  # FWHM = sqrt(8 * log(2)) * sigma
            if np.abs(m2) > delta_min:
                moment_2[i] = fwhm
            else:
                moment_2[i] = 0
        moment_2 = np.nan_to_num(moment_2)
        return moment_2

    moment_2 = calc_moment_2(unique_coords, indices, velo, data)
    # Combine the unique coordinates with their moment 0
    # wrap the result in a sparse matrix (COO format)
    # this is more memory efficient for large datasets
    moment_2_coo = sp.coo_array(
        (moment_2, (unique_coords[:, 0], unique_coords[:, 1])), shape=data[0].shape
    )
    moment_2_coo.eliminate_zeros()  # eliminate zeros to save memory
    return moment_2_coo


# Calculate moment 2 using pure Spectral-Cube (faster for larger clusters)
# https://github.com/radio-astro-tools/spectral-cube/blob/e98b6c3c05e3a21c6ca62524e1dea9582ad5cd38/spectral_cube/_moments.py#L170
def moment_2_sc(data, vyx):
    # create a cube which only contains the values of points in the cluster (labels == cluster_id)
    bool_array = np.zeros(data.shape, dtype=np.bool)
    bool_array[vyx[:, 0], vyx[:, 1], vyx[:, 2]] = (
        True  # mark all points in the cluster (labels == cluster_id) as True
    )
    cluster_data = (
        bool_array * data
    )  # multiply the data with the boolean array to get only the values of points in the cluster
    bool_cube = sc(cluster_data, wcs=wcs).with_spectral_unit(u.km / u.s)
    # Calculate moment 2 (FWHM) of the velocity axis
    moment_2 = bool_cube.linewidth_fwhm().value
    moment_2 = np.nan_to_num(moment_2)
    moment_2[np.abs(moment_2) < delta_min] = 0  # set small values to 0
    # wrap the result in a sparse matrix (COO format)
    # this is more memory efficient for large datasets
    moment_2_coo = sp.coo_array(moment_2)
    return moment_2_coo


def moment_2_func(data, index, labels, n_clusters):
    # Optimized moment_2_func with parallel processing for faster computation.
    def compute_moment_2_for_cluster(cluster_id):
        vyx = index[
            labels == cluster_id
        ]  # Get all coordinates [v, y, x] of points in the cluster (labels == cluster_id)
        size = len(vyx)
        if size < 100000:
            return moment_2_py(
                data, vyx
            )  # Use Python implementation for smaller clusters
        else:
            return moment_2_sc(data, vyx)  # Use spectral-cube for larger clusters

    # Use parallel processing to compute moment 2 for each cluster
    moment_2_results = Parallel(n_jobs=-1, backend="threading")(
        delayed(compute_moment_2_for_cluster)(cluster_id)
        for cluster_id in trange(n_clusters)
    )

    # Combine results into a single array
    moment_2_cube = np.hstack(moment_2_results)
    return moment_2_cube

In [ ]:
moment_2_cube = moment_2_func(data, index, labels, n_clusters)

# 筛选符合要求的 HVC 候选体

SNR = 3, 4, 5

SIZE = 4, 6, 9 pixels  # 3.4, 4.2, 5.1 arcmin

FWHM = 3, 4, 5 km/s

In [ ]:
# Select clusters based on the criteria
# Minimum SNR, SIZE, and FWHM
MIN_SNR = 5
MIN_SIZE = 0  # pixels
MIN_FWHM = 3.5

In [ ]:
# Calculate condition SNR, SIZE, and FWHM for each cluster
def snr_size_fwhm(cluster_id, data, index, labels, MEDIAN, STD, moment_2_cube):
    # vyx is coordinates [v, y, x] of points in the cluster (labels == cluster_id)
    vyx = index[labels == cluster_id]
    cluster_value = data[vyx[:, 0], vyx[:, 1], vyx[:, 2]]
    # Calculate SNR
    peak = np.max(cluster_value)  # peak value in the cluster
    snr = (peak - MEDIAN) / STD  # signal-to-noise ratio (SNR) of the cluster
    # Calculate SIZE (spatial pixels)
    coords = vyx[:, 1:]  # coordinates [y, x]
    unique_yx = np.unique(coords, axis=0)  # unique [y, x] coordinates in the cluster
    # calculate the size of the cluster based on the number of spatial pixels
    size = len(unique_yx)
    # Get FWHM from moment_2_cube
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error")
            fwhm = np.nanmean(moment_2_cube[cluster_id].data)
    except RuntimeWarning as e:
        warnings.simplefilter("ignore", category=RuntimeWarning)
        fwhm = np.nanmean(moment_2_cube[cluster_id].data)  # fwhm, warning (m2 < 0)
    snr_size_fwhm_arr = np.array([snr, size, fwhm])
    snr_size_fwhm_arr = np.nan_to_num(snr_size_fwhm_arr)  # replace NaN with 0
    return snr_size_fwhm_arr


# Find HVC candidates based on SNR, SIZE, and FWHM
def find_hvc_candidates(
    data,
    index,
    labels,
    n_clusters,
    MEDIAN,
    STD,
    moment_2_cube,
    MIN_SNR,
    MIN_SIZE,
    MIN_FWHM,
):
    # calculate condition SNR, SIZE, and FWHM for each cluster in parallel
    snr_size_fwhm_arr = Parallel(n_jobs=-1, backend="threading")(
        delayed(snr_size_fwhm)(
            cluster_id, data, index, labels, MEDIAN, STD, moment_2_cube
        )
        for cluster_id in trange(n_clusters)
    )
    snr_size_fwhm_arr = np.array(snr_size_fwhm_arr)  # convert list to numpy array
    # generate a boolean array based on the conditions
    # True is the index of HVC candidates
    TF = (
        (snr_size_fwhm_arr[:, 0] > MIN_SNR)
        & (snr_size_fwhm_arr[:, 1] > MIN_SIZE)
        & (snr_size_fwhm_arr[:, 2] > MIN_FWHM)
    ).astype(int)
    hvc_candidates = np.nonzero(TF)[0]  # get indices of HVC candidates
    # Create a 2D array with SNR, SIZE, and FWHM for each HVC candidate
    hvc_candidates_array = np.tile(TF, (3, 1)).T * snr_size_fwhm_arr
    SNR = hvc_candidates_array[hvc_candidates][:, 0]
    SIZE = hvc_candidates_array[hvc_candidates][:, 1]
    FWHM = hvc_candidates_array[hvc_candidates][:, 2]
    return hvc_candidates, SNR, SIZE, FWHM


hvc_candidates, SNR, SIZE, FWHM = find_hvc_candidates(
    data,
    index,
    labels,
    n_clusters,
    MEDIAN,
    STD,
    moment_2_cube,
    MIN_SNR,
    MIN_SIZE,
    MIN_FWHM,
)
print(hvc_candidates)

# Number of HVC candidates
N_HVC = len(hvc_candidates)
print("Number of HVC candidates:", N_HVC)

# 使用形状筛选HVC候选体

In [ ]:
def getMinVolEllipse(P=None, tolerance=0.01):
    """Find the minimum volume ellipsoid which holds all the points

    Based on work by Nima Moshtagh
    http://www.mathworks.com/matlabcentral/fileexchange/9542
    and python implementation by
    https://github.com/minillinim/ellipsoid

    Here, P is a numpy array of N dimensional points like this:
    P = [[x,y,z,...], <-- one point per line
         [x,y,z,...],
         [x,y,z,...]]

    Returns:
    (center, radii, rotation)
    """

    (N, d) = np.shape(P)
    # Q will be our working array
    Q = np.vstack([np.copy(P.T), np.ones(N)])
    # initializations
    err = 1.0 + tolerance
    U = np.ones(N) / N
    # Khachiyan Algorithm
    while err > tolerance:
        V = Q @ np.diag(U) @ Q.T  # Weighted covariance, @ is matrix multiplication
        M = Q.T @ np.linalg.inv(V) @ Q  # Mahalanobis distances
        j = np.argmax(M.diagonal())
        maximum = M[j, j]
        step_size = (maximum - d - 1.0) / ((d + 1.0) * (maximum - 1.0))
        new_U = (1.0 - step_size) * U
        new_U[j] += step_size
        err = np.linalg.norm(new_U - U)
        U = new_U
    # center of the ellipse
    center = P.T @ U
    yc, xc = center
    # the A matrix for the ellipse
    A = np.linalg.pinv(P.T @ np.diag(U) @ P - np.outer(center, center)) / d
    # Get the values we'd like to return
    eigenvalues, eigenvectors = np.linalg.eigh(A)
    a, b = 1 / np.sqrt(eigenvalues)  # semi-axis lengths
    theta = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
    theta = -np.degrees(theta)  # Convert theta from radians to degrees
    return xc, yc, a, b, theta

In [ ]:
def fwhm_ellipse(moment_0_crop):
    points_above_half_max = np.argwhere(moment_0_crop > 0.5 * np.max(moment_0_crop))
    points_above_zero = np.argwhere(moment_0_crop > 0)
    area = len(points_above_half_max)  # Number of points above half maximum
    # Calculate the minimum volume ellipse for the included points
    try:
        # 首先尝试使用 points_above_half_max 进行计算
        xc, yc, a, b, theta = getMinVolEllipse(points_above_half_max)
    except np.linalg.LinAlgError as e:
        # 如果上面的计算失败了（例如，点数不足、矩阵奇异等），则捕获异常
        print(f"Singular Matrix Error: {e}. Using all points for ellipse fitting.")
        # 使用备用的 points 数据集进行计算
        xc, yc, a, b, theta = getMinVolEllipse(points_above_zero)
    return xc, yc, a, b, theta, area

In [ ]:
# Calculate the size of the cluster based on the moment 0 cube
def calc_size(cluster_id, moment_0_cube):
    # Get the moment 0 array from the sparse matrix
    moment_0 = moment_0_cube[cluster_id].toarray()
    # Find the non-zero indices in the moment 0 cube
    nonzero_indices = np.nonzero(moment_0)
    y_min, y_max = np.min(nonzero_indices[0]), np.max(nonzero_indices[0])
    x_min, x_max = np.min(nonzero_indices[1]), np.max(nonzero_indices[1])
    moment_0_crop = moment_0[y_min : y_max + 1, x_min : x_max + 1]
    height, width = moment_0_crop.shape
    # Calculate the extent of the cropped moment 0 array
    # note that RA is left bigger, right smaller, and DEC is bottom smaller, top bigger
    extent = [
        wcs.celestial.pixel_to_world_values(x_min, 0)[0],
        wcs.celestial.pixel_to_world_values(x_max, 0)[0],
        wcs.celestial.pixel_to_world_values(0, y_min)[1],
        wcs.celestial.pixel_to_world_values(0, y_max)[1],
    ]
    x_min, x_max, y_min, y_max = extent  # now extent is in world coordinates
    xc, yc, a, b, theta, area = fwhm_ellipse(moment_0_crop)
    x_scale = (x_max - x_min) / (width - 1)  # pixel-to-world scale factor, here 0.025
    y_scale = (y_max - y_min) / (height - 1)  # pixel-to-world scale factor, here 0.025
    xc = x_min + (xc * x_scale)
    yc = y_min + (yc * y_scale)
    area = area * np.abs(x_scale * y_scale)  # area from pixel**2 to deg**2
    a = a * np.abs(x_scale)  # a is the semi-major axis in degrees
    b = b * np.abs(y_scale)  # b is the sami-minor axis in degrees
    return moment_0_crop, extent, xc, yc, a, b, theta, area

In [ ]:
cluster_params = []
ellipse_params = []

for i in range(len(hvc_candidates)):
    cluster_id = hvc_candidates[i]
    # Calculate the fwhm size for the cluster
    moment_0_crop, extent, xc, yc, a, b, theta, area = calc_size(
        cluster_id, moment_0_cube
    )
    # area in contour
    area *= 3600  # area from deg**2 to arcmin**2
    size_arcmin = (
        2 * np.sqrt(a * b) * 60
    )  # size is the average diameter of the ellipse in arcmin
    # area in ellipse
    area2 = np.pi * size_arcmin**2 / 4
    cluster_params.append((moment_0_crop, extent, xc, yc, a, b, theta, area))
    ellipse_params.append((xc, yc, a, b, theta))
    print(f"Cluster {i}: size: {size_arcmin:.2f} arcmin")
    print(f"Area in FWHM Contour: {area:.2f} arcmin^2")
    print(f"Area in Ellipse: {area2:.2f} arcmin^2", "\n")

ellipse_params = np.array(ellipse_params)  # convert list to numpy array

# 使用光谱筛选HVC候选体

In [ ]:
# 将分辨率从 delta_v 平滑到 hi4pi_delta_v
# 计算对应的 sigma（km/s）
factor = 2.0 * np.sqrt(2.0 * np.log(2.0))  # FWHM = factor * sigma
sigma_orig = delta_v / factor
sigma_target = hi4pi_delta_v / factor
sigma_conv = np.sqrt(max(0, sigma_target**2 - sigma_orig**2))
print("sigma_conv =", sigma_conv, "km/s")

# 转换为以“通道数”为单位的卷积标准差
sigma_conv_ch = sigma_conv / delta_v

# 构建高斯核
kernel = Gaussian1DKernel(stddev=sigma_conv_ch)

def smooth_spectrum(crafts_spectrum):
    # 对光谱卷积平滑（使用 FFT 边界处理）
    crafts_spectrum_smoothed_full = convolve_fft(
        crafts_spectrum,
        kernel,
        normalize_kernel=True,
        allow_huge=True,
        preserve_nan=False
    )
    # 插值到目标速度轴
    crafts_spectrum_smoothed = np.interp(
        hi4pi_velocities,
        velocities[::-1],
        crafts_spectrum_smoothed_full
    )
    crafts_spectrum_smoothed = crafts_spectrum_smoothed[::-1]
    return crafts_spectrum_smoothed

In [ ]:
crafts_spectra = []
crafts_spectra_smoothed = []
hi4pi_spectra = []

for i in range(len(hvc_candidates)):
    cluster_id = hvc_candidates[i]
    # vyx is coordinates [v, y, x] of points in the cluster (labels == cluster_id)
    vyx = index[labels == cluster_id]
    bool_array = np.zeros(data.shape, dtype=np.bool_)
    bool_array[vyx[:, 0], vyx[:, 1], vyx[:, 2]] = True
    cluster_data = bool_array * data
    peak_vyx = np.unravel_index(np.argmax(cluster_data), cluster_data.shape)
    peak_coord = wcs.pixel_to_world_values(peak_vyx[2], peak_vyx[1], peak_vyx[0])
    v_min, v_max = velocities[vyx[:, 0].min()], velocities[vyx[:, 0].max()]
    xc, yc, a, b, theta = ellipse_params[i, :]
    center = SkyCoord(xc, yc, unit="deg", frame="fk5")
    region = EllipseSkyRegion(
        center=center, width=a * u.deg, height=b * u.deg, angle=-theta * u.deg
    )
    crafts_subcube = cube.subcube_from_regions([region])
    crafts_subcube_data = crafts_subcube.unmasked_data[:, :, :].value
    crafts_spectrum = np.mean(crafts_subcube_data, axis=(1, 2))
    if len(crafts_spectrum) == 0:
        print("CRAFTS: Empty subcube for cluster", i)
        y_peak, x_peak = peak_vyx[1], peak_vyx[2]
        crafts_spectrum = cube[:, y_peak, x_peak].value
        peak_coord_sky = SkyCoord(peak_coord[0], peak_coord[1], unit="deg", frame="fk5")
        hi4pi_wcs = hi4pi_cube.wcs.celestial
        peak_coord_hi4pi = hi4pi_wcs.world_to_pixel(peak_coord_sky)
        hi4pi_spectrum = hi4pi_cube[
            :, int(peak_coord_hi4pi[1]), int(peak_coord_hi4pi[0])
        ].value
    else:
        hi4pi_subcube = hi4pi_cube.subcube_from_regions([region])
        hi4pi_subcube_data = hi4pi_subcube.unmasked_data[:, :, :].value
        hi4pi_spectrum = np.mean(hi4pi_subcube_data, axis=(1, 2))
        if len(hi4pi_spectrum) == 0:
            print("HI4PI: Empty subcube for cluster", i)
            y_peak, x_peak = peak_vyx[1], peak_vyx[2]
            crafts_spectrum = cube[:, y_peak, x_peak].value
            peak_coord_sky = SkyCoord(peak_coord[0], peak_coord[1], unit="deg", frame="fk5")
            hi4pi_wcs = hi4pi_cube.wcs.celestial
            peak_coord_hi4pi = hi4pi_wcs.world_to_pixel(peak_coord_sky)
            hi4pi_spectrum = hi4pi_cube[
                :, int(peak_coord_hi4pi[1]), int(peak_coord_hi4pi[0])
            ].value
    crafts_spectrum_smoothed = smooth_spectrum(crafts_spectrum)
    print(len(crafts_spectrum), len(crafts_spectrum_smoothed), len(hi4pi_spectrum))
    crafts_spectra.append(crafts_spectrum)
    crafts_spectra_smoothed.append(crafts_spectrum_smoothed)
    hi4pi_spectra.append(hi4pi_spectrum)

# 生成形状和光谱的图像

In [ ]:
ID = []

for i in trange(len(hvc_candidates)):
    cluster_id = hvc_candidates[i]
    # ellipse parameters
    xc, yc, a, b, theta = ellipse_params[i]
    # coordinates
    c_eq = SkyCoord(xc, yc, frame="fk5", unit="deg")
    c_gal = c_eq.galactic
    glon = c_gal.l.value
    glat = c_gal.b.value
    # vyx is coordinates [v, y, x] of points in the cluster (labels == cluster_id)
    vyx = index[labels == cluster_id]
    bool_array = np.zeros(data.shape, dtype=np.bool_)
    bool_array[vyx[:, 0], vyx[:, 1], vyx[:, 2]] = True
    cluster_data = bool_array * data
    # peak brightness temperature
    tpkb = np.max(cluster_data)
    # find the peak velocity
    peak_vyx = np.unravel_index(np.argmax(cluster_data), cluster_data.shape)
    peak_coord = wcs.pixel_to_world_values(peak_vyx[2], peak_vyx[1], peak_vyx[0])
    # calculate vlsr and vgsr
    vlsr = peak_coord[2] / 1000  # km/s
    hvc_id = f"HVC{glon:.2f}{'+' if glat >= 0 else '-'}{np.abs(glat):.2f}{'+' if vlsr >= 0 else '-'}{vlsr:.0f}"
    ID.append(hvc_id)

In [ ]:
for i in range(len(hvc_candidates)):
    cluster_id = hvc_candidates[i]
    moment_0_crop, extent, xc, yc, a, b, theta, area = cluster_params[i]
    fig, ax = plt.subplots(1, 2, figsize=[10, 4])
    im = ax[0].imshow(
        moment_0_crop,
        extent=extent,
        origin="lower",
        cmap="viridis_r",
        interpolation="none",
    )
    cs = ax[0].contour(
        moment_0_crop,
        levels=[0.5 * np.max(moment_0_crop)],
        extent=extent,
        colors="white",
        linewidths=1,
        linestyles="solid",
    )
    ellipse = Ellipse(
        (xc, yc),
        width=2 * a,
        height=2 * b,
        angle=90 - theta,
        edgecolor="red",
        facecolor="none",
        lw=2,
    )
    ax[0].add_patch(ellipse)
    ax[0].set_xlabel("Right Ascension (FK5)")
    ax[0].set_ylabel("Declination (FK5)")
    ax[0].set_title(f"Cluster")
    ax[0].grid(color="white", ls="--", alpha=0.5)
    plt.colorbar(im, ax=ax[0], label="Moment 0 (K km/s)")
    ax[1].plot(velocities, crafts_spectra[i], lw=0.5, alpha=0.5, label="CRAFTS")
    ax[1].plot(hi4pi_velocities, crafts_spectra_smoothed[i], lw=1.5, label="CRAFTS smoothed")
    ax[1].plot(hi4pi_velocities, hi4pi_spectra[i], lw=1.5, label="HI4PI")
    ax[1].set_xlabel("Velocity LSR (km/s)")
    ax[1].set_ylabel("Flux Density (K)")
    ax[1].set_title(f"Spectrum")
    fig.tight_layout()
    fig.suptitle(f"{ID[i]}", fontsize=16)
    plt.legend()
    plt.savefig(f"./Figures/{ID[i]}.png", dpi=300)
    plt.show()

# 最后生成HVC源表

In [ ]:
# HVC catalog DataFrame
# ID, RA, DEC, GLON, GLAT, VLSR, VGSR, SIZEa, SIZEb, FWHM, SNR, TPKB, N_HI
CATALOG = pd.DataFrame(
    columns=[
        "ID",
        "RA",
        "DEC",
        "GLON",
        "GLAT",
        "VLSR",
        "VGSR",
        "SIZEa",
        "SIZEb",
        "SNR",
        "FWHM",
        "TPKB",
        "N_HI",
    ]
)

ID = []

for i in trange(len(hvc_candidates)):
    cluster_id = hvc_candidates[i]
    # ellipse parameters
    xc, yc, a, b, theta = ellipse_params[i]
    a_arcmin = 2 * a * 60  # major axis in arcmin
    b_arcmin = 2 * b * 60  # minor axis in arcmin
    # coordinates
    c_eq = SkyCoord(xc, yc, frame="fk5", unit="deg")
    c_gal = c_eq.galactic
    ra = c_eq.ra.value
    dec = c_eq.dec.value
    glon = c_gal.l.value
    glat = c_gal.b.value
    # vyx is coordinates [v, y, x] of points in the cluster (labels == cluster_id)
    vyx = index[labels == cluster_id]
    bool_array = np.zeros(data.shape, dtype=np.bool_)
    bool_array[vyx[:, 0], vyx[:, 1], vyx[:, 2]] = True
    cluster_data = bool_array * data
    # peak brightness temperature
    tpkb = np.max(cluster_data)
    # find the peak velocity
    peak_vyx = np.unravel_index(np.argmax(cluster_data), cluster_data.shape)
    peak_coord = wcs.pixel_to_world_values(peak_vyx[2], peak_vyx[1], peak_vyx[0])
    # calculate vlsr and vgsr
    vlsr = peak_coord[2] / 1000  # km/s
    vgsr = vlsr + 220 * np.sin(np.deg2rad(glon)) * np.cos(np.deg2rad(glat))
    # HI column density
    # MNRAS 432, 3074–3079 (2013)
    n_hi = np.max(moment_0_cube[cluster_id]) * 1.823 / 100  # 1e20 cm^-2
    # SNR and FWHM
    snr = SNR[i]
    fwhm = FWHM[i]
    # HVC candidate ID
    # The ID of the cloud. in the traditional form for HVCs,
    # obtained from the galactic coordinates at the nominal cloud center
    # and the V_LSR of the cloud
    hvc_id = f"HVC{glon:.2f}{'+' if glat >= 0 else '-'}{np.abs(glat):.2f}{'+' if vlsr >= 0 else '-'}{vlsr:.0f}"
    ID.append(hvc_id)
    # fill the catalog
    CATALOG.loc[i] = {
        "ID": hvc_id,
        "RA": ra,
        "DEC": dec,
        "GLON": glon,
        "GLAT": glat,
        "VLSR": vlsr,
        "VGSR": vgsr,
        "SIZEa": a_arcmin,
        "SIZEb": b_arcmin,
        "SNR": snr,
        "FWHM": fwhm,
        "TPKB": tpkb,
        "N_HI": n_hi,
    }

In [ ]:
# CATALOG = CATALOG.sort_values(by=CATALOG.columns[0])
print(CATALOG)

In [ ]:
false_candidates = []  # indices of false candidates
false_candidates_id = [ID[i] for i in false_candidates]
print(false_candidates_id)

CATALOG_NEW = CATALOG[~CATALOG[CATALOG.columns[0]].isin(false_candidates_id)]

In [ ]:
# save 2D morphology and spectra from CRAFTS and HI4PI to text file for each HVC candidate
# both true and false candidates
for i in range(len(hvc_candidates)):
    id = ID[i]
    np.savetxt(f'./crafts_spectrum/{id}.txt', crafts_spectra_smoothed[i], fmt='%.6f')
    np.savetxt(f'./hi4pi_spectrum/{id}.txt', hi4pi_spectra[i], fmt='%.6f')
    np.savetxt(f'./2D_morph/{id}.txt', cluster_params[i][0], fmt='%.6f')

In [ ]:
CATALOG_NEW.round(2)

In [ ]:
hvc_candidates_filter = np.ones_like(hvc_candidates, dtype=bool)
hvc_candidates_filter[false_candidates] = False

hvc_candidates_filtered = hvc_candidates[hvc_candidates_filter]
ellipse_params_filtered = ellipse_params[hvc_candidates_filter]

In [ ]:
vyx_clusters = np.empty((0, 3))
for cluster_id in hvc_candidates_filtered:
    # vyx is coordinates [v, y, x] of points in the cluster (labels == cluster_id)
    vyx = index[labels == cluster_id]
    vyx_clusters = np.vstack((vyx_clusters, vyx))
vyx_clusters = vyx_clusters.astype(int)

bool_array = np.zeros(data.shape, dtype=np.bool_)
# mark all points in the clusters (labels == cluster_id) as True
bool_array[vyx_clusters[:, 0], vyx_clusters[:, 1], vyx_clusters[:, 2]] = True

# all HVC candidates cube
# if replace `bool_array` with `mask`, the data before DBSCAN will be displayed
hvc_candidates_cube = sc(bool_array * data, wcs=wcs).with_spectral_unit(u.km / u.s)

# elliptical apertures for all HVC candidates
aper = [
    SkyEllipticalAperture(
        SkyCoord(ra=xc * u.deg, dec=yc * u.deg),
        a * u.deg,
        b * u.deg,
        theta=theta * u.deg,
    )
    for xc, yc, a, b, theta in ellipse_params_filtered
]

In [ ]:
fig, ax = plt.subplots(
    1,
    3,
    figsize=[15, 4],
    sharex=True,
    sharey=True,
    layout="compressed",
    subplot_kw={"projection": wcs.celestial},
)

moment_0 = hvc_candidates_cube.moment(order=0)
norm = (
    None
    if np.all(np.isnan(moment_0.value))
    else simple_norm(moment_0.value, percent=95)
)
im0 = ax[0].imshow(moment_0.value, norm=norm, cmap="viridis_r", origin="lower")
for aper_i, txt in zip(aper, hvc_candidates_filtered):
    aper_to_pixels = aper_i.to_pixel(wcs.celestial)
    aper_to_pixels.plot(ax=ax[0], color="red", lw=1, label=f"Cluster {txt}")
ax[0].grid(linestyle="--")
ax[0].set_title("Moment 0 (K km/s)")
ax[0].set_xlabel("RA")
ax[0].set_ylabel("DEC")
lon = ax[0].coords[0]
lat = ax[0].coords[1]
lon.set_major_formatter("dd")
lat.set_major_formatter("dd")
lon.set_axislabel("RA")
lat.set_axislabel("Dec")
plt.colorbar(im0, ax=ax[0])

moment_1 = hvc_candidates_cube.moment(order=1)
norm = (
    None
    if np.all(np.isnan(moment_1.value))
    else simple_norm(moment_1.value, percent=95)
)
im1 = ax[1].imshow(moment_1.value, norm=norm, cmap="viridis_r", origin="lower")
for aper_i, txt in zip(aper, hvc_candidates_filtered):
    aper_to_pixels = aper_i.to_pixel(wcs.celestial)
    aper_to_pixels.plot(ax=ax[1], color="red", lw=1, label=f"Cluster {txt}")
ax[1].grid(linestyle="--")
ax[1].set_title("Moment 1 (km/s)")
lon = ax[1].coords[0]
lat = ax[1].coords[1]
lon.set_major_formatter("dd")
lat.set_major_formatter("dd")
lon.set_axislabel("RA")
lat.set_axislabel("Dec")
plt.colorbar(im1, ax=ax[1])

moment_2 = hvc_candidates_cube.linewidth_fwhm()
norm = (
    None
    if np.all(np.isnan(moment_2.value))
    else simple_norm(moment_2.value, percent=95)
)
im2 = ax[2].imshow(moment_2.value, norm=norm, cmap="viridis_r", origin="lower")
for aper_i, txt in zip(aper, hvc_candidates_filtered):
    aper_to_pixels = aper_i.to_pixel(wcs.celestial)
    aper_to_pixels.plot(ax=ax[2], color="red", lw=1, label=f"Cluster {txt}")
ax[2].grid(linestyle="--")
ax[2].set_title("FWHM (km/s)")
lon = ax[2].coords[0]
lat = ax[2].coords[1]
lon.set_major_formatter("dd")
lat.set_major_formatter("dd")
lon.set_axislabel("RA")
lat.set_axislabel("Dec")
plt.colorbar(im2, ax=ax[2])

fig.suptitle(
    f"eps = {np.round(eps, 3)}, minPts = {minPts}, MIN_SNR = {MIN_SNR}, "
    f"MIN_SIZE = {MIN_SIZE}, MIN_FWHM = {MIN_FWHM}"
)
plt.show()

moment_0.write(output_file[0], overwrite=True)
moment_1.write(output_file[1], overwrite=True)
moment_2.write(output_file[2], overwrite=True)

# 记录表

eps = 1, sqrt(2), sqrt(3)

min_samples = 2, 4, 6,  
              6, 9, 12, 15, 18,
              6, 9, 12, 15, 18, 19

**记录每一个(eps, min_samples)参数组下，表现最好的SNR/spatial_pixels/FWHM**

N为HVC candidate数目，F为其中的假信号数目，T为其中的真信号数目

| eps | minPts |     | SNR | FWHM(km/s) |     | N   | T   | F   |
| --- | ------ | --- | --- | ---------- | --- | --- | --- | --- |
| 3   | 19     |     | 6.5 | 6          |     | 16  | 16  | 0   |
| 3   | 18     |     | 5   | 4          |     | 15  | 15  | 0   |
| 3   | 15     |     | 5   | 3.5        |     | 17  | 14  | 3   |
| 3   | 12     |     | 5   | 3.5        |     | 21  | 17  | 4   |
| 3   | 9      |     | 4.5 | 3.5        |     | 20  | 16  | 4   |
| 3   | 6      |     | 5   | 3.5        |     | 20  | 16  | 4   |
|     |        |     |     |            |     |     |     |     |
| 2   | 18     |     | 5   | 4          |     | 14  | 14  | 0   |
| 2   | 15     |     | 5   | 3.5        |     | 17  | 14  | 3   |
| 2   | 12     |     | 5   | 3.5        |     | 21  | 17  | 4   |
| 2   | 9      |     | 4.5 | 4          |     | 20  | 16  | 4   |
| 2   | 6      |     | 5   | 3.5        |     | 20  | 16  | 4   |
|     |        |     |     |            |     |     |     |     |
| 1   | 6      |     | 5   | 3.5        |     | 19  | 15  | 4   |
| 1   | 4      |     | 4.5 | 3.5        |     | 19  | 16  | 3   |
| 1   | 2      |     | 5   | 3.5        |     | 19  | 16  | 3   |


```
all_false_hvc_candidates = [
    # RA = 0, sign = -
    'HVC199.14-39.20--82', 'HVC199.50-31.25--58', 'HVC197.79-33.79--57', 
    'HVC196.18-36.17--74', 'HVC202.58-26.54--77', 'HVC197.55-34.15--55', 
    'HVC197.20-34.63--56', 'HVC199.10-31.75--57', 'HVC197.97-40.41--61', 
    'HVC196.58-35.58--87', 'HVC202.58-26.54--90', 'HVC201.60-36.07--180', 
    'HVC196.25-34.53--191', 'HVC201.96-26.06--200', 'HVC194.49-36.86--192',
    
    # RA = 1, sign = -
    'HVC211.97-11.03--59', 'HVC208.90-15.40--81', 'HVC213.97-5.54--103', 
    'HVC211.76-5.85--115', 'HVC206.07-18.98--252',
    
    # RA = 2, sign = -
    'HVC221.51+5.98--71', 'HVC215.71-0.46--251', 'HVC217.51+3.10--252', 
    'HVC219.10+6.16--252',
    
    # RA = 3, sign = -
    'HVC235.91+24.73--86', 'HVC228.50+28.98--75', 'HVC233.48+21.12--82', 
    'HVC234.71+14.63--208', 'HVC224.97+16.83--251',
    
    # RA = 0, sign = +
    'HVC200.20-36.66+268', 'HVC202.81-32.05+202', 'HVC200.18-36.69+119', 
    'HVC206.88-27.51+101', 'HVC203.59-33.19+83', 'HVC204.37-31.28+88', 
    'HVC208.94-24.49+100', 'HVC201.79-35.94+89', 'HVC207.81-25.39+87', 
    'HVC209.05-24.89+89', 'HVC199.15-39.16+87', 'HVC202.90-34.43+83', 
    'HVC208.50-23.92+90', 'HVC208.80-24.16+89', 'HVC201.30-35.94+85', 
    'HVC203.58-32.88+93', 'HVC205.30-31.02+84', 'HVC200.30-37.65+88', 
    'HVC197.97-40.41+91',
    
    # RA = 1, sign = +
    'HVC208.88-19.07+139',
    
    # RA = 2, sign = +
    'HVC217.70+0.09+292', 'HVC218.92+5.70+272', 'HVC218.79+5.44+259', 
    'HVC222.40+12.31+252',
    
    # RA = 3, sign = +
    'HVC240.84+21.50+296'
]
```